## NaCl Qeq-SOG 时间对比：直接求解 vs 迭代（PCG）

本 notebook 用 `CACE-SOG-Qeq2` 里的 `ChargeEq`，在相同的 NaCl 结构上对比：

- **direct**：`use_iterative_solver=False`（显式构造 A + `torch.linalg.solve`）
- **iter**：`use_iterative_solver=True`（SOG 算子 + PCG + Schur 消元）

只计 Qeq 的前向时间（不训练），用 `fit-4hdnnp-NaCl/NaCl.xyz` 中前若干帧做 benchmark。

In [1]:
import os, sys, time
from pathlib import Path

import torch
from ase.io import read


# 假定本 notebook 位于 CACE-SOG-Qeq2/fit-4hdnnp-NaCl-compare/
ROOT_DIR = Path("/work/home/acrb3qk4vo/SOG-Qeq/SOG-Net/CACE-SOG-Qeq2")
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import cace
from cace.modules import ChargeEq

print("ROOT_DIR =", ROOT_DIR)
print("CUDA available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

ROOT_DIR = /work/home/acrb3qk4vo/SOG-Qeq/SOG-Net/CACE-SOG-Qeq2
CUDA available: False


device(type='cpu')

In [7]:
# 读取 NaCl 结构（NaCl.xyz 在 fit-4hdnnp-NaCl 下）
ROOT = Path(ROOT_DIR)
data_dir = ROOT / "fit-4hdnnp-NaCl"
xyz_path = data_dir / "NaCl.xyz"
assert xyz_path.exists(), f"NaCl.xyz not found at {xyz_path}"

atoms_list = read(str(xyz_path), ":")  # 读取所有帧
len(atoms_list)

5000

In [8]:
# 构造两个 ChargeEq：direct 和 iter（仅测试 Qeq 部分，不依赖上游网络）

elements = [11, 17]  # Na, Cl
sog_num_components = 18

charge_eq_direct = ChargeEq(
    dl=1.5,
    sigma=1.0,
    elements=elements,
    feature_key="chi",
    output_key="q_eq",
    ewald_key="SOG_potential",
    system_charge=None,
    remove_self_interaction=True,
    aggregation_mode="sum",
    use_iterative_solver=False,
    use_sog_kernel=True,
    sog_num_components=sog_num_components,
)

charge_eq_iter = ChargeEq(
    dl=1.5,
    sigma=1.0,
    elements=elements,
    feature_key="chi",
    output_key="q_eq",
    ewald_key="SOG_potential",
    system_charge=None,
    remove_self_interaction=True,
    aggregation_mode="sum",
    use_iterative_solver=True,
    max_cg_iters=200,
    cg_tol=1e-6,
    use_sog_kernel=True,
    sog_num_components=sog_num_components,
)

charge_eq_direct.to(device)
charge_eq_iter.to(device)

charge_eq_direct.eval()
charge_eq_iter.eval()

ChargeEq(
  (ep): EwaldPotential()
)

In [2]:
def build_qeq_input(atoms, device):
    """给定单个 ASE Atoms，构造 ChargeEq.forward 需要的 data dict。"""
    pos = torch.as_tensor(atoms.get_positions(), dtype=torch.get_default_dtype(), device=device)
    cell = torch.as_tensor(atoms.cell.array.reshape(1, 3, 3), dtype=torch.get_default_dtype(), device=device)
    Z = torch.as_tensor(atoms.get_atomic_numbers(), dtype=torch.long, device=device)
    N = pos.shape[0]

    # 这里用一个简单的 chi：全零即可（不影响时间复杂度）
    chi = torch.zeros((N, 1), dtype=torch.get_default_dtype(), device=device)

    # 假设整体电中性
    system_charge = torch.zeros(1, dtype=torch.get_default_dtype(), device=device)

    return {
        "positions": pos,
        "cell": cell,
        "batch": None,
        "chi": chi,
        "atomic_numbers": Z,
        "system_charge": system_charge,
    }


# 防止用户跳过上面 cell 导致 NameError：若未定义则在此构造
if "charge_eq_direct" not in globals() or "charge_eq_iter" not in globals():
    elements = [11, 17]
    sog_num_components = 18
    charge_eq_direct = ChargeEq(
        dl=1.5,
        sigma=1.0,
        elements=elements,
        feature_key="chi",
        output_key="q_eq",
        ewald_key="SOG_potential",
        system_charge=None,
        remove_self_interaction=True,
        aggregation_mode="sum",
        use_iterative_solver=False,
        use_sog_kernel=True,
        sog_num_components=sog_num_components,
    ).to(device)
    charge_eq_iter = ChargeEq(
        dl=1.5,
        sigma=1.0,
        elements=elements,
        feature_key="chi",
        output_key="q_eq",
        ewald_key="SOG_potential",
        system_charge=None,
        remove_self_interaction=True,
        aggregation_mode="sum",
        use_iterative_solver=True,
        max_cg_iters=200,
        cg_tol=1e-6,
        use_sog_kernel=True,
        sog_num_components=sog_num_components,
    ).to(device)
    charge_eq_direct.eval()
    charge_eq_iter.eval()


# 简单测试一帧是否能跑通（若未先运行「读取 NaCl」cell，则在此加载）
if "atoms_list" not in globals() or len(globals().get("atoms_list", [])) == 0:
    data_dir = ROOT_DIR / "fit-4hdnnp-NaCl"
    xyz_path = data_dir / "NaCl.xyz"
    atoms_list = read(str(xyz_path), ":")

test_data = build_qeq_input(atoms_list[0], device)
with torch.no_grad():
    out_direct = charge_eq_direct(test_data.copy())
    out_iter = charge_eq_iter(test_data.copy())

out_direct["q_eq"].shape, out_iter["q_eq"].shape

NameError: name 'charge_eq_direct' is not defined

In [ ]:
# 对前 n_structures 个 NaCl 结构做时间对比
n_structures = 20  # 可根据需要调整
subset = atoms_list[:n_structures]

def time_qeq(module, atoms_seq):
    torch.cuda.empty_cache() if device.type == "cuda" else None
    start = time.perf_counter()
    with torch.no_grad():
        for at in atoms_seq:
            data = build_qeq_input(at, device)
            _ = module(data)
    end = time.perf_counter()
    return (end - start) / len(atoms_seq)

# 预热
_ = time_qeq(charge_eq_direct, subset[:2])
_ = time_qeq(charge_eq_iter, subset[:2])

t_direct = time_qeq(charge_eq_direct, subset)
t_iter = time_qeq(charge_eq_iter, subset)

print(f"Avg time per structure (direct solve): {t_direct*1000:.3f} ms")
print(f"Avg time per structure (iterative PCG): {t_iter*1000:.3f} ms")
print(f"Speedup (direct / iter): {t_direct / t_iter if t_iter>0 else float('nan'):.2f}x")